# LoanShield: Data Preprocessing
This notebook handles:
1. Data Cleaning (Removing hidden spaces).
2. **Outlier Removal** (Using IQR method).
3. Encoding (Binary, Ordinal, One-Hot).
4. Scaling & Splitting.

**Output:** Processed CSV files and a saved Scaler object.

In [11]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Configuration
pd.set_option('display.max_columns', None)

In [13]:
# 2. Load Data
file_path = "../data/raw/loan_default_raw.csv"
df = pd.read_csv(file_path)

# --- SAFETY CHECK 1 ---
print(f"Initial Shape: {df.shape}")
print(f"Initial Default=1 Count: {df['Default'].sum()}")

Initial Shape: (255347, 18)
Initial Default=1 Count: 29653


## Part 1: Data Cleaning
- Dropping irrelevant columns (`LoanID`).
- Checking for duplicates/nulls again (just to be safe).

In [ ]:

# Drop irrelevant ID column
if 'LoanID' in df.columns:
    df = df.drop('LoanID', axis=1)

# Clean hidden spaces in text columns (Crucial Bug Fix)
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].str.strip()

print("✅ ID dropped and string spaces cleaned.")

✅ ID dropped and string spaces cleaned.


In [ ]:
# 4. Outlier Removal (IQR Method)
# We focus on numerical columns where extreme values might distort the model.
# Note: We avoid removing outliers from 'Default' (Target) or binary flags.

numerical_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed']

print(f"Shape before outlier removal: {df.shape}")

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter: Keep rows within bounds
    old_count = df.shape[0]
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    new_count = df.shape[0]
    
    if old_count - new_count > 0:
        print(f"Dropped {old_count - new_count} outliers from {col}")

print(f"Shape after outlier removal: {df.shape}")
print(f"Remaining Default=1 Count: {df['Default'].sum()}")

Shape before outlier removal: (255347, 17)
Shape after outlier removal: (255347, 17)
Remaining Default=1 Count: 29653


## Part 2: Feature Encoding
We need to convert text columns into numbers for the Machine Learning models.

**Strategy:**
1. **Binary Encoding (0/1):** For columns with Yes/No (`HasMortgage`, `HasDependents`, `HasCoSigner`).
2. **Ordinal Encoding:** For `Education` (High School < Bachelor's < Master's < PhD).
3. **One-Hot Encoding:** For categorical nominal variables (`EmploymentType`, `MaritalStatus`, `LoanPurpose`).

In [ ]:

# A. Binary Encoding (Yes/No)
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    # Safety Check
    if df[col].isnull().sum() > 0:
        raise ValueError(f"NaNs created in {col} during mapping!")

# B. Ordinal Encoding (Education)
education_map = {
    'High School': 0, 
    "Bachelor's": 1, 
    "Master's": 2, 
    'PhD': 3
}
df['Education'] = df['Education'].map(education_map)

# C. One-Hot Encoding (Nominal Categories)
# dtype=int ensures we get 0/1 instead of True/False
df = pd.get_dummies(df, columns=['EmploymentType', 'MaritalStatus', 'LoanPurpose'], drop_first=True, dtype=int)

print("✅ Encoding Complete.")

✅ Encoding Complete.


## Part 3: Train-Test Split
We split the data **before** scaling to prevent data leakage.
- **Train Set:** 80% (Used to train models)
- **Test Set:** 20% (Used to evaluate performance)
- **Stratify:** Ensures the proportion of Defaults (1s) is the same in both sets.

In [ ]:
# We split BEFORE scaling to prevent data leakage.

X = df.drop('Default', axis=1)
y = df['Default']

# Stratify=y ensures we keep the same ratio of Defaulters in Train and Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train Shape: {X_train.shape}")
print(f"Test Shape: {X_test.shape}")

Train Shape: (204277, 22)
Test Shape: (51070, 22)


## Part 4: Feature Scaling
We use `StandardScaler` to bring numerical features to a similar range (mean=0, std=1).
This is crucial for algorithms like Logistic Regression.

In [ ]:
# We scale all numerical features using StandardScaler

cols_to_scale = [
    'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 
    'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio'
]

scaler = StandardScaler()

# Fit on TRAIN, transform TEST
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])



## Part 5: Saving Files
We save the processed datasets and the scaler object for future use.

In [19]:
# Save Files
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../webapp/models', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
joblib.dump(scaler, '../webapp/models/scaler.pkl')

print("✅ Preprocessing Done. Files Saved.")

✅ Preprocessing Done. Files Saved.
